In [ ]:
import torch
import torchvision
import os
from ldm.models.diffusion.ddpm import LatentDiffusion
from ldm.models.diffusion.ddim import DDIMSampler
from dataset import SimpleHandsDataset
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

# --- PARAMÈTRES ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_images = 2000
n_epochs = 5
batch_size = 4
lr = 1e-5
save_dir = "./results_diffusion"
os.makedirs(save_dir, exist_ok=True)

# --- CONFIGURATION DU MODÈLE ---
unet_config = {
    "target": "ldm.modules.diffusionmodules.openaimodel.UNetModel",
    "params": {
        "image_size": 64, 
        "in_channels": 4,
        "out_channels": 4,
        "model_channels": 128,
        "attention_resolutions": [4, 2, 1],
        "num_res_blocks": 2,
        "channel_mult": [1, 2, 4, 4],
        "num_heads": 8,
    }
}

first_stage_config = {
    "target": "ldm.models.autoencoder.AutoencoderKL",
    "params": {
        "embed_dim": 4,
        "monitor": "val/rec_loss",
        "ckpt_path": "vae_hands_latest.pt", 
        "ddconfig": {
            "double_z": True, "z_channels": 4, "resolution": 256,
            "in_channels": 3, "out_ch": 3, "ch": 128,
            "ch_mult": [1, 2, 4, 4], "num_res_blocks": 2, "attn_resolutions": []
        },
        "lossconfig": {"target": "torch.nn.Identity"}
    }
}

# 1. INSTANCIATION DU MODÈLE
print("Initialisation du modèle Latent Diffusion...")
model = LatentDiffusion(
    unet_config=unet_config,
    first_stage_config=first_stage_config,
    cond_stage_config="__is_unconditional__", 
    timesteps=1000,
    beta_schedule="linear",
    linear_start=0.00085,
    linear_end=0.0120,
    conditioning_key=None
).to(device)

# Correction pour s'assurer que les buffers sont sur le bon device
model.logvar = model.logvar.to(device)

# Initialisation du Sampler DDIM pour la génération
sampler = DDIMSampler(model)
sampler.make_schedule(ddim_num_steps=50, ddim_eta=0.0, verbose=False)

# 2. PRÉPARATION DES DONNÉES
print("Chargement du dataset...")
full_dataset = SimpleHandsDataset(root_dir="./data/hands")

if len(full_dataset) > max_images:
    indices = torch.randperm(len(full_dataset))[:max_images]
    full_dataset = torch.utils.data.Subset(full_dataset, indices)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)

# 3. OPTIMISEUR
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# 4. BOUCLE D'ENTRAÎNEMENT
print(f"Début de l'entraînement pour {n_epochs} epochs...")

for epoch in range(n_epochs):
    # --- PHASE TRAIN ---
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}")
    for batch in pbar:
        img = batch.to(device)
        
        # Encodage latent (Espace VAE)
        with torch.no_grad():
            latents = model.get_first_stage_encoding(model.encode_first_stage(img))
        
        # Diffusion : on prédit le bruit
        t = torch.randint(0, model.num_timesteps, (latents.shape[0],), device=device).long()
        loss, _ = model.p_losses(latents, None, t) 
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    # --- PHASE VALIDATION ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for v_batch in val_loader:
            v_img = v_batch.to(device)
            # On encode en latent
            v_latents = model.get_first_stage_encoding(model.encode_first_stage(v_img))
            v_t = torch.randint(0, model.num_timesteps, (v_latents.shape[0],), device=device).long()
            # Calcul de la perte
            v_l, _ = model.p_losses(v_latents, None, v_t)
            val_loss += v_l.item()
            
    avg_val_loss = val_loss / len(val_loader)
    print(f" --- Val Loss: {avg_val_loss:.4f} ---")

    # --- GÉNÉRATION DE TEST ---
    print(f"Génération d'images de test...")
    with torch.no_grad():
        # Bruit pur (4 canaux, 64x64)
        shape = [4, 64, 64] 
        # Sampling DDIM (50 étapes au lieu de 1000)
        samples_latent, _ = sampler.sample(S=50, conditioning=None, batch_size=4, shape=shape, verbose=False)
        # Décodage Latent -> Pixel
        samples_pixel = model.decode_first_stage(samples_latent)
        
        # Normalisation et sauvegarde
        samples_pixel = torch.clamp((samples_pixel + 1.0) / 2.0, min=0.0, max=1.0)
        save_path = os.path.join(save_dir, f"sample_epoch_{epoch+1}.png")
        torchvision.utils.save_image(samples_pixel, save_path, nrow=2)
        print(f"Image sauvegardée : {save_path}")

    # Sauvegarde du modèle
    torch.save(model.state_dict(), "latent_diffusion_hands.pt")

print("Entraînement terminé !")

Initialisation du modèle Latent Diffusion...
LatentDiffusion: Running in eps-prediction mode
DiffusionWrapper has 102.77 M params.
Keeping EMAs of 368.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels


C:\Users\thiba\Dev\genia\projet_zeta\ldm\models\autoencoder.py:318: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(path, map_location="cpu")


Restored from vae_hands_latest.pt
Training LatentDiffusion as an unconditional model.
Chargement du dataset...
Début de l'entraînement pour 5 epochs...


Epoch 1/5: 100%|████████████████████████████████████████████████████████| 400/400 [28:58<00:00,  4.35s/it, loss=0.5454]


 --- Val Loss: 0.5108 ---
Génération d'images de test...
Data shape for DDIM sampling is (4, 4, 64, 64), eta 0.0
Running DDIM Sampling with 50 timesteps


...

Epoch 2/5: 100%|████████████████████████████████████████████████████████| 400/400 [35:11<00:00,  5.28s/it, loss=0.3110]


 --- Val Loss: 0.3097 ---
Génération d'images de test...
Data shape for DDIM sampling is (4, 4, 64, 64), eta 0.0
Running DDIM Sampling with 50 timesteps


DDIM Sampler: 100%|████████████████████████████████████████████████████████████████████| 50/50 [02:51<00:00,  3.43s/it]


Image sauvegardée : ./results_diffusion\sample_epoch_2.png


Epoch 3/5:   2%|█▏                                                        | 8/400 [00:42<34:12,  5.24s/it, loss=0.2999]